# 家庭記帳：微調 Qwen2.5-1.5B 做記帳解析（加分項）

上線用的是商業 API（見 `docs/03-LLM實作規劃.md`）。這本筆記本是「我們自己微調一顆小模型」的對照組：
跑完會得到一個 GGUF 檔、放上 Hugging Face，之後可以用 `backend/tools/eval_nlp.py` 跟商業 API 比成績。

**開始前（只要做一次）**

1. 上方選單 **執行階段 → 變更執行階段類型 → T4 GPU**
2. 左邊的 🔑（Secrets）新增 `HF_TOKEN`，值是 Hugging Face 的 **write** token
   （huggingface.co → Settings → Access Tokens → New token → Write）。**不要把 token 貼在程式碼裡。**
3. 改下面第一格的 `HF_REPO`：`你的帳號/qwen2.5-1.5b-fambudget-GGUF`

然後 **執行階段 → 全部執行**。T4 上大約 30～50 分鐘（訓練 20～30 分鐘、轉 GGUF 10 分鐘左右）。

In [ ]:
HF_REPO = "你的帳號/qwen2.5-1.5b-fambudget-GGUF"   # ← 改成自己的
BASE = "unsloth/Qwen2.5-1.5B-Instruct"
EPOCHS = 2
MAX_LEN = 2048

## 1. 安裝（2～3 分鐘）

In [ ]:
!pip install -q unsloth

## 2. 拿訓練資料

訓練資料由 repo 裡的 `ml/make_data.py`、`ml/build_sft.py` 產生，已經放在 `ml/data/`。
repo 是公開的就直接 clone；不是的話，把 `ml/data/sft.jsonl` 跟 `ml/data/tools.json` 拖進左邊的檔案區。

In [ ]:
import json, os, shutil, subprocess

if not os.path.exists("sft.jsonl"):
    subprocess.run(["git", "clone", "--depth", "1", "https://github.com/cbes502034/final_project.git"])
    for name in ("sft.jsonl", "tools.json"):
        src = os.path.join("final_project", "ml", "data", name)
        if os.path.exists(src):
            shutil.copy(src, name)
assert os.path.exists("sft.jsonl") and os.path.exists("tools.json"), "把 sft.jsonl 跟 tools.json 拖進左邊的檔案區再跑一次"

rows = [json.loads(line) for line in open("sft.jsonl", encoding="utf-8") if line.strip()]
tools = json.load(open("tools.json", encoding="utf-8"))
print(len(rows), "段對話；驗證集", sum(r["split"] == "val" for r in rows))

## 3. 載入模型（4-bit）並掛上 LoRA

In [ ]:
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(BASE, max_seq_length=MAX_LEN, load_in_4bit=True)
model = FastLanguageModel.get_peft_model(
    model,
    r=16, lora_alpha=16, lora_dropout=0, bias="none",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    use_gradient_checkpointing="unsloth", random_state=3407,
)

## 4. 套對話模板

**一定要帶 `tools=tools`**：上線時 llama-server（`--jinja`）會用同一個模板把工具定義排進 system。
訓練跟上線看到的 prompt 不一樣，微調就白做了。

In [ ]:
from datasets import Dataset

def render(row):
    return tokenizer.apply_chat_template(row["messages"], tools=tools, tokenize=False)

train = Dataset.from_dict({"text": [render(r) for r in rows if r["split"] == "train"]})
val = Dataset.from_dict({"text": [render(r) for r in rows if r["split"] == "val"]})
longest = max(len(tokenizer(t).input_ids) for t in train["text"][:300])
print("最長約", longest, "tokens（要小於 MAX_LEN）")
print(train[0]["text"][-700:])

## 5. 訓練（只學 assistant 的回答，不學 prompt）

In [ ]:
from trl import SFTConfig, SFTTrainer
from unsloth.chat_templates import train_on_responses_only

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,          # 新版 trl 若報錯，改成 processing_class=tokenizer
    train_dataset=train,
    eval_dataset=val,
    args=SFTConfig(
        dataset_text_field="text",
        max_seq_length=MAX_LEN,
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        warmup_steps=10,
        num_train_epochs=EPOCHS,
        learning_rate=2e-4,
        logging_steps=10,
        eval_strategy="steps",
        eval_steps=50,
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="linear",
        seed=3407,
        output_dir="outputs",
        report_to="none",
    ),
)
trainer = train_on_responses_only(trainer, instruction_part="<|im_start|>user\n",
                                  response_part="<|im_start|>assistant\n")
stats = trainer.train()
print(stats)

## 6. 先試一題（驗證集的第一題）

輸出應該是 `<tool_call>{"name": "record_transactions", "arguments": {...}}</tool_call>`。
截圖存起來，報告的「訓練結果」用得到；訓練過程的 loss 也一起截。

In [ ]:
FastLanguageModel.for_inference(model)
sample = next(r for r in rows if r["split"] == "val")
inputs = tokenizer.apply_chat_template(sample["messages"][:2], tools=tools, add_generation_prompt=True,
                                       return_tensors="pt").to("cuda")
out = model.generate(input_ids=inputs, max_new_tokens=600, do_sample=False)
print(sample["messages"][1]["content"].split("輸入：")[-1])
print(tokenizer.decode(out[0][inputs.shape[1]:], skip_special_tokens=False))

## 7. 轉成 GGUF（q4_k_m）並推上 Hugging Face

會先編譯 llama.cpp，10 分鐘左右。完成後在 `https://huggingface.co/<HF_REPO>` 看得到 `.gguf` 檔。

In [ ]:
from google.colab import userdata

token = userdata.get("HF_TOKEN")
model.push_to_hub_gguf(HF_REPO, tokenizer, quantization_method="q4_k_m", token=token)
print("好了：https://huggingface.co/" + HF_REPO)

## 8. 接下來

1. 照 `ml/space/README.md` 開一個 Hugging Face Space，把模型跑成 OpenAI 相容的服務
2. `backend/.env` 改成：`MODEL_PROVIDER=openai`、`MODEL_BASE_URL=<Space 網址>`、`MODEL_NAME=qwen2.5-1.5b-fambudget`
3. 在 `backend/` 底下跑 `python tools/eval_nlp.py --label qwen-ft`，再用 `--compare` 跟商業 API、規則比